In [ ]:
import torch
import torch.nn as nn
from torchrl.envs.utils import step_mdp, check_env_specs
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.data.replay_buffers import ReplayBuffer, LazyTensorStorage
from torchrl.modules import SafeModule, ProbabilisticActor, ValueOperator
from torchrl.modules.distributions import NormalParamExtractor, TanhNormal
from torchrl.objectives.sac import SACLoss
from torchrl.objectives import SoftUpdate, group_optimizers
 
from ksp_state import KSPState
from qvn_model import QVNModel
 
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")

For this RL problem I will be using the SAC (Soft Actor Critic) algorithm. This is an algorithm that finds the best stochastic policy: pi, and the best Q(s,a) function. We do this by approximating their values though two neural networks: theta and phi. So we estimate pi(.|s ; phi) and Q(s,a | theta). We then use a distribution to choose our actions. The second crucial part of the algorithm is that the objective function: J(theta) includes a "entropy term" which is maximized. Meaning that the model is allowed to explore the space that it is given.

Kerbal Space Program is a continious problem, I thought about making this a sequential problem by allowing "frames" where the agent would act (say every 4 frames), as in the original DQN paper. Yet Kerbal space program needs inputs that are continious such as thrust (eg sometime I need the rockets thrust to be 10% to land and not intermitent from 100% to 10%). So I discarded DQC, DDQC, and other competing algorithms for SAC. Note that the implementation of SAC that I will be using (from torchRL) was chosen due to already implementing hyperparameter optimization and two competing Q-networks, which we choose the minimum value of to minimize training time.

At saying this our primary job is to implement an enviorment using KRPC that is compatible with torchRL's enviorments. This way in the future, and maybe even within the same project we could get to the point where we can implement and compare many algorithms. A modified enviorment might still allow DDQN, PPO and more.

DAC came from both robotics, as it is generally the most popular algorithm in the area but also a series of youtube videos about training an RL agent to beat world record in the racing game: trackmania. There was an offhand mention of the algorithm in one of the videos so I decided to investigate and came upon the original DAC paper and code in github.

In [ ]:
env = KSPState(
    target_orbit_altitude=80_000,
    step_interval=0.5,
    max_steps=2000,
)

N_OBS = 10
N_ACTIONS = 3
HIDDEN = 256



In [ ]:
actor_net = nn.Sequential(
    nn.Linear(N_OBS, HIDDEN),
    nn.ReLU(),
    nn.Linear(HIDDEN, HIDDEN),
    nn.ReLU(),
    nn.Linear(HIDDEN, 2 * N_ACTIONS),
    NormalParamExtractor(),
)

actor_module = SafeModule(
    actor_net,
    in_keys=["observation"],
    out_keys=["loc", "scale"],
)
 
actor = ProbabilisticActor(
    module=actor_module,
    in_keys=["loc", "scale"],
    out_keys=["action"],
    spec=env.action_spec,
    distribution_class=TanhNormal,
).to(device)
 


In [ ]:
qvalue = ValueOperator(
    module=QVNModel(N_OBS, N_ACTIONS, HIDDEN),
    in_keys=["observation", "action"],
).to(device)

Here we can leverage the compatibility of the state created with the torchRL package. Thus we can use the built in SAC algorithm, and specifically use the two q-networks as noted by 1801.01290v2

In [ ]:
loss_module = SACLoss(
    actor_network=actor,
    qvalue_network=qvalue,
    num_qvalue_nets=2,
    gamma=0.99,
    alpha_init=0.2,
    target_entropy="auto",
    loss_function="smooth_l1",
)
loss_module.to(device)

target_updater = SoftUpdate(loss_module, tau=0.005)


optimizer_actor = torch.optim.Adam(
    loss_module.actor_network_params.values(True, True), lr=3e-4
)
optimizer_critic = torch.optim.Adam(
    loss_module.qvalue_network_params.values(True, True), lr=3e-4
)
optimizer_alpha = torch.optim.Adam(
    [loss_module.log_alpha], lr=3e-4
)
optimizer = group_optimizers(optimizer_actor, optimizer_critic, optimizer_alpha)
del optimizer_actor, optimizer_critic, optimizer_alpha

In [ ]:
buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=100_000),
    batch_size=256,
)

In [ ]:
BATCH_SIZE = 256
NUM_EPISODES = 500
WARMUP_STEPS = 1000
UPDATES_PER_STEP = 1
 
total_steps = 0
 
for episode in range(NUM_EPISODES):
    td = env.reset()
    episode_reward = 0.0
    episode_steps = 0
 
    while True:
        if total_steps < WARMUP_STEPS:
            td["action"] = env.action_spec["action"].rand()
        else:
            td_device = td.to(device)
            with torch.no_grad(), set_exploration_type(ExplorationType.RANDOM):
                td_device = actor(td_device)
            td["action"] = td_device["action"].cpu()
            td["loc"] = td_device["loc"].cpu()
            td["scale"] = td_device["scale"].cpu()
        

        next_td = env.step(td)
 
        episode_reward += next_td["next", "reward"].item()
        episode_steps += 1
        total_steps += 1

        buffer.add(next_td)

        if total_steps >= WARMUP_STEPS and len(buffer) >= BATCH_SIZE:
            for _ in range(UPDATES_PER_STEP):
                batch = buffer.sample().to(device)
 
                loss_td = loss_module(batch)
 
                total_loss = (
                    loss_td["loss_actor"]
                    + loss_td["loss_qvalue"]
                    + loss_td["loss_alpha"]
                )
 
                optimizer.zero_grad(set_to_none=True)
                total_loss.backward()
                optimizer.step()
 
                target_updater.step()

        if next_td["next", "done"].item():
            break

        td = step_mdp(next_td)

        alpha = loss_module.alpha.item() if total_steps >= WARMUP_STEPS else 0.2
    if episode > 0 and episode % 50 == 0:
            torch.save({
                'episode': episode,
                'total_steps': total_steps,
                'loss_module': loss_module.state_dict(),
                'optimizer': optimizer.state_dict(),
            }, f"checkpoint_ep{episode}.pt")
            print(f"Saved checkpoint_ep{episode}.pt")
    print(
        f"Episode {episode:4d} | "
        f"steps: {episode_steps:5d} | "
        f"total: {total_steps:7d} | "
        f"reward: {episode_reward:8.2f} | "
        f"alpha: {alpha:.4f} | "
        f"orbit: {'YES' if env._orbit_achieved else 'no'}"
    )

In [ ]:
env.close()